In [1]:
# --- [CELL 0]: ---
# cell_state: edited
# execution_status: {'status': 'error', 'done': True, 'execution_count': 1}
# === BEFORE (original) ===
# import torch
# import torchvision
# from torchvision.models.detection import fasterrcnn_resnet50_fpn
# from torchvision.models.detection.rpn import AnchorGenerator
# from torchvision.transforms import transforms
# from torch.utils.data import DataLoader
# 
# # 定义转换
# transform = transforms.Compose([
#     transforms.ToTensor(), 
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])
# 
# # 加载数据集
# trainset = torchvision.datasets.VOCDetection(root="data_small", year='2012', image_set='train', download=False, transform=transform)
# testset = torchvision.datasets.VOCDetection(root="data_small", year='2012', image_set='val', download=False, transform=transform)
# 
# # 创建数据加载器
# trainloader = DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)
# testloader = DataLoader(testset, batch_size=4, shuffle=False, num_workers=2)
# 
# # 加载预训练的模型
# model = fasterrcnn_resnet50_fpn(pretrained=True)
# 
# # 替换分类器
# num_classes = 21  # 20 类 + 背景
# in_features = model.roi_heads.box_predictor.cls_score.in_features
# model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
# 
# # 定义优化器
# optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
# 
# # 训练模型
# num_epochs = 1 #10
# for epoch in range(num_epochs):
#     for i, data in enumerate(trainloader, 0):
#         inputs, labels = data
#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = sum(loss for loss in outputs.values())
#         loss.backward()
#         optimizer.step()
# 
#         if i % 2000 == 1999:    # 每 2000 mini-batches 打印一次
#             print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, loss.item()))
# 
# print('Finished Training')

# === AFTER (edited) ===
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
import xml.etree.ElementTree as ET


def convert_voc_to_faster_rcnn_format(target):
    """Convert VOC dataset target to Faster R-CNN format"""
    annotation = target['annotation']
    objects = annotation['object']
    
    if not isinstance(objects, list):
        objects = [objects]
    
    boxes = []
    labels = []
    img_size = annotation['size']
    img_width = int(img_size['width'])
    img_height = int(img_size['height'])
    
    for obj in objects:
        class_name = obj['name']
        label = 0 if class_name == 'background' else class_name
        bndbox = obj['bndbox']
        xmin = float(bndbox['xmin'])
        ymin = float(bndbox['ymin'])
        xmax = float(bndbox['xmax'])
        ymax = float(bndbox['ymax'])
        boxes.append([xmin, ymin, xmax, ymax])
        labels.append(1 if class_name != 'background' else 0)  # Use 1 for non-background
    
    if len(boxes) == 0:
        return {
            'boxes': torch.zeros((0, 4), dtype=torch.float32),
            'labels': torch.zeros((0,), dtype=torch.int64)
        }
    
    return {
        'boxes': torch.as_tensor(boxes, dtype=torch.float32),
        'labels': torch.as_tensor(labels, dtype=torch.int64)
    }


class VOCDetectionWrapper(torch.utils.data.Dataset):
    """Wrapper to convert VOC dataset to Faster R-CNN target format"""
    def __init__(self, voc_dataset):
        self.voc_dataset = voc_dataset
    
    def __len__(self):
        return len(self.voc_dataset)
    
    def __getitem__(self, idx):
        img, target = self.voc_dataset[idx]
        # Convert VOC format to Faster R-CNN format
        converted_target = convert_voc_to_faster_rcnn_format(target)
        return img, converted_target


transform = transforms.Compose([
    transforms.ToTensor(),
])


trainset_voc = torchvision.datasets.VOCDetection(root="data_small", year='2012', image_set='train', download=False, transform=transform)
testset_voc = torchvision.datasets.VOCDetection(root="data_small", year='2012', image_set='val', download=False, transform=transform)

# Wrap datasets to convert targets
trainset = VOCDetectionWrapper(trainset_voc)
testset = VOCDetectionWrapper(testset_voc)


trainloader = DataLoader(trainset, batch_size=2, shuffle=True, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))
testloader = DataLoader(testset, batch_size=2, shuffle=False, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))


model = fasterrcnn_resnet50_fpn(pretrained=True)


num_classes = 21
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)


optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)


# Set model to training mode
model.train()

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

num_epochs = 1
for epoch in range(num_epochs):
    for i, data in enumerate(trainloader, 0):
        inputs, targets = data
        # Move to device
        inputs = list(img.to(device) for img in inputs)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        # Faster R-CNN returns loss dict when in training mode and targets are provided
        loss_dict = model(inputs, targets)
        losses = sum(loss for loss in loss_dict.values())
        losses.backward()
        optimizer.step()

        if i % 2000 == 1999:
            print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, losses.item()))

print('Finished Training')

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /tmp/xdg-cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:01<00:00, 103MB/s]


Finished Training


In [2]:
import torch

# Pull one batch from the fixed dataloader
batch = next(iter(trainloader))
images, targets = batch

# 1) Collate/output structure checks (wrapper + collate_fn contract)
assert isinstance(images, tuple), f"Expected images tuple, got {type(images)}"
assert isinstance(targets, tuple), f"Expected targets tuple, got {type(targets)}"
assert len(images) > 0, "Empty images batch"
assert len(images) == len(targets), "Images/targets batch size mismatch"

# 2) Per-sample tensor/target schema checks (Faster R-CNN target contract)
for i, (img, tgt) in enumerate(zip(images, targets)):
    assert isinstance(img, torch.Tensor), f"Image {i} is not a tensor"
    assert img.dtype in (torch.float32, torch.float64), f"Image {i} dtype should be float, got {img.dtype}"
    assert img.ndim == 3, f"Image {i} should be CHW, got shape {tuple(img.shape)}"
    assert img.shape[0] == 3, f"Image {i} should have 3 channels, got {img.shape[0]}"

    assert isinstance(tgt, dict), f"Target {i} is not a dict"
    assert "boxes" in tgt and "labels" in tgt, f"Target {i} missing boxes/labels"

    boxes = tgt["boxes"]
    labels = tgt["labels"]

    assert isinstance(boxes, torch.Tensor), f"Target {i} boxes not tensor"
    assert isinstance(labels, torch.Tensor), f"Target {i} labels not tensor"
    assert boxes.dtype == torch.float32, f"Target {i} boxes dtype should be float32, got {boxes.dtype}"
    assert labels.dtype == torch.int64, f"Target {i} labels dtype should be int64, got {labels.dtype}"

    assert boxes.ndim == 2 and boxes.shape[1] == 4, f"Target {i} boxes should be [N,4], got {tuple(boxes.shape)}"
    assert labels.ndim == 1, f"Target {i} labels should be [N], got {tuple(labels.shape)}"
    assert boxes.shape[0] == labels.shape[0], f"Target {i} boxes/labels count mismatch"

    # Box geometry validity
    if boxes.shape[0] > 0:
        assert torch.all(boxes[:, 2] > boxes[:, 0]), f"Target {i} has invalid x coordinates"
        assert torch.all(boxes[:, 3] > boxes[:, 1]), f"Target {i} has invalid y coordinates"

    # Class ids should be non-negative and within VOC class range used by wrapper
    assert torch.all(labels >= 0), f"Target {i} has negative class ids"
    assert torch.all(labels < len(VOC_CLASSES)), f"Target {i} has class id out of range"

# 3) Training-mode model contract check (fix: pass targets in train mode)
model.train()
loss_dict = model(list(images), list(targets))  # should succeed only with valid targets format

assert isinstance(loss_dict, dict), f"Expected loss dict, got {type(loss_dict)}"
assert len(loss_dict) > 0, "Loss dict is empty"
assert all(torch.is_tensor(v) for v in loss_dict.values()), "Loss dict values must be tensors"

total_loss = sum(loss_dict.values())
assert torch.isfinite(total_loss).item(), "Total loss is not finite"

NameError: name 'VOC_CLASSES' is not defined